# Sentiment Analysis and Named Entity Recognition

## Overview

Sentiment analysis classifies the emotional tone of text; Named Entity Recognition (NER) identifies and classifies named entities (locations, measurements, chemical species). Both are core applied NLP tasks.

**Sentiment approaches:**

| Approach | Tool | Pros | Cons |
|---|---|---|---|
| Lexicon-based | VADER | No training data needed | Domain mismatch |
| ML-based | sklearn + TF-IDF | Trainable on domain data | Needs labels |
| Transformer-based | HuggingFace | State-of-the-art | Compute heavy |

**NER approaches:**
- Rule-based: regex patterns for measurements, site codes
- Statistical: spaCy pre-trained models
- Fine-tuned: best for domain-specific entity types

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

rng = np.random.default_rng(42)
labelled_texts = [
    ("Species richness increased substantially following buffer restoration.", 1),
    ("Nitrate levels dropped 40% after constructed wetland installation.", 1),
    ("Macroinvertebrate communities recovering toward reference condition.", 1),
    ("Water quality significantly improved at all downstream monitoring sites.", 1),
    ("EPT richness doubled within two seasons of riparian planting.", 1),
    ("Fish populations rebounding after passage barrier removal.", 1),
    ("Phosphorus concentrations within acceptable range for three months.", 1),
    ("Vegetation cover restored; erosion rates reduced by 60%.", 1),
    ("Elevated nitrate detected at all six monitoring sites this quarter.", 0),
    ("Phosphorus exceeded threshold; algal bloom risk increased.", 0),
    ("Macroinvertebrate index declined to poor status at downstream sites.", 0),
    ("Dissolved oxygen critically low; fish kill reported in lower reach.", 0),
    ("Turbidity elevated following storm event; data flagged unreliable.", 0),
    ("Species richness at lowest recorded level since monitoring began.", 0),
    ("Agricultural runoff detected; nutrient loading increased 30%.", 0),
    ("pH below acceptable range indicating continued acidification stress.", 0),
]
texts_raw  = [t for t,l in labelled_texts]
labels_raw = np.array([l for t,l in labelled_texts])
print(f"{len(texts_raw)} labelled documents: {labels_raw.sum()} positive, {(labels_raw==0).sum()} negative")

16 labelled documents: 8 positive, 8 negative


---
## Lexicon-Based Sentiment with VADER

In [2]:
try:
    import nltk
    nltk.download('vader_lexicon', quiet=True)
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    sia = SentimentIntensityAnalyzer()
    print("VADER sentiment (compound: -1=most negative, +1=most positive):")
    print(f"{'Text':52s} {'Compound':>10} {'True':>6}")
    correct = 0
    for text, label in labelled_texts:
        scores = sia.polarity_scores(text)
        comp   = scores['compound']
        pred   = 1 if comp >= 0.05 else 0
        correct += (pred == label)
        match = 'OK' if pred == label else 'XX'
        print(f"{text[:51]:52s} {comp:10.3f} {label:6d} {match}")
    print(f"\nVADER accuracy: {correct}/{len(labelled_texts)} = {correct/len(labelled_texts):.0%}")
    print("VADER trained on social media -- may misread technical language")
except ImportError:
    print('pip install nltk && python -c "import nltk; nltk.download(\'vader_lexicon\')"')
    print("VADER: returns neg/neu/pos/compound scores per text")

VADER sentiment (compound: -1=most negative, +1=most positive):
Text                                                   Compound   True
Species richness increased substantially following        0.649      1 OK
Nitrate levels dropped 40% after constructed wetlan       0.000      1 XX
Macroinvertebrate communities recovering toward ref       0.000      1 XX
Water quality significantly improved at all downstr       0.477      1 OK
EPT richness doubled within two seasons of riparian       0.494      1 OK
Fish populations rebounding after passage barrier r      -0.128      1 XX
Phosphorus concentrations within acceptable range f       0.318      1 OK
Vegetation cover restored; erosion rates reduced by       0.340      1 OK
Elevated nitrate detected at all six monitoring sit       0.000      0 OK
Phosphorus exceeded threshold; algal bloom risk inc       0.000      0 OK
Macroinvertebrate index declined to poor status at       -0.477      0 OK
Dissolved oxygen critically low; fish kill reported

---
## ML Sentiment Classifier

In [3]:
positive_pool = [
    "restoration improved water quality significantly",
    "species richness increased after intervention",
    "macroinvertebrates recovered to reference condition",
    "buffer strips successfully reduced nitrate loading",
    "ecological recovery observed at all sites",
    "biodiversity indices show positive trend",
]
negative_pool = [
    "nitrate exceeded safe limits at monitoring stations",
    "water quality deteriorated during monitoring period",
    "algal bloom detected upstream sampling point",
    "macroinvertebrate diversity at critically low level",
    "phosphorus runoff increased agricultural drainage",
    "species richness declined this monitoring quarter",
]
all_texts  = texts_raw + positive_pool*3 + negative_pool*3
all_labels = np.array(labels_raw.tolist() + [1]*18 + [0]*18)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True,
                               stop_words='english', max_features=300)),
    ('clf',   LogisticRegression(C=1.0, max_iter=500,
                                  class_weight='balanced', random_state=42)),
])
scores = cross_val_score(pipe, all_texts, all_labels, cv=cv, scoring='f1')
print(f"ML sentiment CV F1: {scores.mean():.3f} +/- {scores.std():.3f}")
pipe.fit(all_texts, all_labels)
new_texts = [
    "Water quality monitoring revealed concerning nitrate trends.",
    "Species richness improved following restoration activities.",
]
preds = pipe.predict(new_texts)
probs = pipe.predict_proba(new_texts)[:,1]
for text, pred, prob in zip(new_texts, preds, probs):
    print(f"\n{text}")
    print(f"  -> {'positive' if pred==1 else 'negative'} (P(pos)={prob:.3f})")

ML sentiment CV F1: 0.904 +/- 0.064

Water quality monitoring revealed concerning nitrate trends.
  -> negative (P(pos)=0.458)

Species richness improved following restoration activities.
  -> positive (P(pos)=0.615)


---
## Rule-Based NER for Ecological Text

In [4]:
def extract_entities_ecological(text):
    entities = []
    # Measurements with units
    for val, unit in re.findall(
            r'(\d+\.?\d*)\s*(mg/[Ll]|mg\.L-1|ug/[Ll]|degC|%|ppm|ppb)', text):
        entities.append(('MEASUREMENT', f'{val} {unit}'))
    # Site codes (e.g. NORTH-04, SITE_012)
    for code in re.findall(r'\b([A-Z]{2,6}[-_]\d{2,4})\b', text):
        entities.append(('SITE_CODE', code))
    # Year ranges / dates
    for date in re.findall(r'\b(\d{4}[-]\d{4}|\d{4}-\d{2}-\d{2})\b', text):
        entities.append(('DATE', date))
    # Chemical parameters
    chems = re.findall(
        r'\b(nitrate|phosphorus|turbidity|pH|dissolved oxygen|ammonia|'
        r'conductivity|temperature|BOD|COD)\b', text, re.IGNORECASE)
    for chem in sorted(set(c.lower() for c in chems)):
        entities.append(('CHEMICAL', chem))
    return entities

test_sentences = [
    "Nitrate levels reached 8.5 mg/L at site NORTH-04 during 2022-03.",
    "Phosphorus concentration was 3.2 mg/L; temperature measured at SOUTH-12.",
    "EPT richness improved from 12 to 28 taxa at EAST-07 (2021-2023).",
    "pH dropped below 6.0; dissolved oxygen at 4.1 mg/L at WEST-03.",
]
for sent in test_sentences:
    ents = extract_entities_ecological(sent)
    print(f"Text: {sent}")
    for ent_type, value in ents:
        print(f"  [{ent_type}] {value}")
    print()

Text: Nitrate levels reached 8.5 mg/L at site NORTH-04 during 2022-03.
  [MEASUREMENT] 8.5 mg/L
  [SITE_CODE] NORTH-04
  [CHEMICAL] nitrate

Text: Phosphorus concentration was 3.2 mg/L; temperature measured at SOUTH-12.
  [MEASUREMENT] 3.2 mg/L
  [SITE_CODE] SOUTH-12
  [CHEMICAL] phosphorus
  [CHEMICAL] temperature

Text: EPT richness improved from 12 to 28 taxa at EAST-07 (2021-2023).
  [SITE_CODE] EAST-07
  [DATE] 2021-2023

Text: pH dropped below 6.0; dissolved oxygen at 4.1 mg/L at WEST-03.
  [MEASUREMENT] 4.1 mg/L
  [SITE_CODE] WEST-03
  [CHEMICAL] dissolved oxygen
  [CHEMICAL] ph



In [5]:
# spaCy NER for standard entity types
try:
    import spacy
    try:
        nlp = spacy.load('en_core_web_sm')
    except OSError:
        raise ImportError('Model not found')
    test_text = (
        'The River Dee restoration project (2021-2023) conducted by Environment Agency '
        'Scotland showed significant improvement. Monitoring at 12 sites across Aberdeen '
        'catchment revealed nitrate reductions of 35%.'
    )
    doc = nlp(test_text)
    print('spaCy NER on ecological report text:')
    for ent in doc.ents:
        print(f'  [{ent.label_:12s}] {ent.text}')
    print('\nNote: en_core_web_sm not trained on ecological text.')
    print('Domain entities (site codes, chemical measurements) need custom training.')
except (ImportError, Exception):
    print('spaCy NER: pip install spacy && python -m spacy download en_core_web_sm')
    print('Standard types: PERSON, ORG, GPE, DATE, CARDINAL, QUANTITY')
    print('For ecological NER, fine-tune on annotated domain text')

spaCy NER on ecological report text:
  [DATE        ] 2021-2023
  [CARDINAL    ] 12
  [NORP        ] Aberdeen
  [PERCENT     ] 35%

Note: en_core_web_sm not trained on ecological text.
Domain entities (site codes, chemical measurements) need custom training.


---

## Common Pitfalls

**1. Applying VADER to scientific or technical text without validation**  
VADER was calibrated on social media and product reviews. Technical language like "nitrate exceeded threshold" may score as neutral because it lacks emotive vocabulary. Always validate lexicon-based tools against a domain-labelled sample before trusting their output.

**2. Treating sentiment as binary when the text is mixed**  
A monitoring report may describe both improvements and concerns in the same document. A single positive/negative label loses this nuance. Consider sentence-level sentiment or a continuous score rather than a hard binary label.

**3. Building a regex NER system without handling case and variant spellings**  
"nitrate", "Nitrate", "NITRATE", and "nitrate-N" are the same entity. Patterns matching only one form will miss the others. Always use `re.IGNORECASE` and include common variants; test against a diverse sample of real documents.

**4. Using a general-purpose spaCy model for domain entity extraction without fine-tuning**  
spaCy's `en_core_web_sm` recognises people, places, and organisations but will not recognise "EPT richness", "NORTH-04", or "8.5 mg/L" as domain entities. For reliable domain NER, annotate a training set and fine-tune.

**5. Evaluating NER on entity type accuracy without checking span boundaries**  
A model that correctly identifies "nitrate" as CHEMICAL but misses the measurement "8.5 mg/L" immediately following has only partial NER correctness. Always evaluate using span-exact match (both entity type and exact span must agree) and report F1 per entity type.

---
*python_methods_library - Samantha McGarrigle*